## Milestone 3
This milestone is designed to familiarize you with the core mechanics of Retrieval-Augmented Generation (RAG). You will learn how to query a vector index for relevant context and use a Cross-Encoder to re-rank the results for maximum accuracy. You will then run side-by-side A/B tests to compare a model's zero-shot inference with and without this context. Finally, we will explore the physical limits of context windows (tuning the correct number of chunks to retrieve) and demonstrate the catastrophic dangers of feeding an LLM incorrect data.

In [ ]:
!pip install kagglehub

Defaulting to user installation because normal site-packages is not writeable


In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md


# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
from pathlib import Path

print(Path.cwd())

for root, dirs, files in os.walk("."):
    if "train.csv" in files:
        print("FOUND:", root)

/home/studinstru/dlGenAI/dlgenai-t22026/notebooks


Step 1: Setup & environment check

In [3]:
import sys, platform
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers

import matplotlib.pyplot as plt

print("Python      :", sys.version.split()[0])
print("Platform    :", platform.system(), platform.release())
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("-" * 45)

# The device we'll use for all tensor computation.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device      :", device)
if device == "cuda":
    print("GPU         :", torch.cuda.get_device_name(0))
    print("VRAM (GB)   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print("-" * 45)

# Confirm the dataset files exist (paths are relative to this notebook's folder).
DATA_DIR = Path("../dataset/smart-mcq-solver-challenge")
for fname in ["train.csv", "test.csv", "sample_submission.csv"]:
    fpath = DATA_DIR / fname
    status = "FOUND   " if fpath.exists() else "MISSING!"
    print(f"{fname:22s} {status} -> {fpath}")

#import os
#from kaggle_secrets import UserSecretsClient
#os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

/home/studinstru/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/studinstru/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Python      : 3.10.12
Platform    : Linux 6.6.114.1-microsoft-standard-WSL2
NumPy       : 2.2.6
Pandas      : 2.3.3
PyTorch     : 2.5.1+cu121
Transformers: 5.13.0
---------------------------------------------
Device      : cuda
GPU         : NVIDIA GeForce RTX 3050 Laptop GPU
VRAM (GB)   : 4.29
---------------------------------------------
train.csv              FOUND    -> ../dataset/smart-mcq-solver-challenge/train.csv
test.csv               FOUND    -> ../dataset/smart-mcq-solver-challenge/test.csv
sample_submission.csv  FOUND    -> ../dataset/smart-mcq-solver-challenge/sample_submission.csv


Step 2: Reading inputs

In [4]:
train = pd.read_csv("../dataset/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("../dataset/smart-mcq-solver-challenge/test.csv")
sample_sub = pd.read_csv("../dataset/smart-mcq-solver-challenge/sample_submission.csv")

sample_sub.to_csv("submission.csv", index=False)

In [5]:
test.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [6]:
train.describe()

,id
count,2000.000000
mean,1000.500000
std,577.494589
min,1.000000
25%,500.750000
50%,1000.500000
75%,1500.250000
max,2000.000000


In [7]:
# check the shape of train and test
print(train.shape)
print(test.shape)
print(sample_sub.shape)

(2000, 8)
(500, 7)
(500, 2)


In [8]:
print("train columns:", list(train.columns))
print("test columns :", list(test.columns))
print()

print("Missing values per column (train):")
print(train.isna().sum().to_string())
print()

train columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
test columns : ['id', 'prompt', 'A', 'B', 'C', 'D', 'E']

Missing values per column (train):
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0



### Milestone 3 setup (run first, before answering any questions)
Run this code in a code cell
 before answering any of the question. This will create your knowledge 
base and the FAISS index for this milestone.

faiss-cpu is the CPU version of FAISS (Facebook AI Similarity Search), a library developed by Meta Platforms for fast similarity search and clustering of dense vectors (embeddings).

Why do we need FAISS?

When you have thousands or millions of vectors (embeddings), comparing a query vector with every vector one by one is very slow.

FAISS creates an efficient index so that it can quickly find the nearest neighbors (most similar vectors).

In [9]:
!pip install faiss-cpu #install FAISS

Defaulting to user installation because normal site-packages is not writeable


In [9]:
import pandas as pd     # Read and manipulate CSV files
import numpy as np      # Numerical operations
import faiss            # Fast similarity search between embeddings
# SentenceTransformer : Convert text to embeddings using pre-trained models
# CrossEncoder : Compute similarity scores between pairs of sentences and rerank them
from sentence_transformers import SentenceTransformer, CrossEncoder
# hugging face model hub to load pre-trained models and tokenizers
from transformers import AutoTokenizer, pipeline 
# TF-IDF Vectorizer : Convert text to numerical vectors based on term frequency and inverse document frequency
from sklearn.feature_extraction.text import TfidfVectorizer 
# Cosine Similarity : Measure similarity between two vectors based on the cosine of the angle between them
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('../dataset/smart-mcq-solver-challenge/train.csv') 

print("Creating knowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
print("Knowledge base embeddings shape:", kb_embeddings.shape) # 2000 x 384

# "Flat" means store all vectors exactly as they are (no compression or approximation).
# "L2" means similarity is measured using the Euclidean (L2) distance
# Smaller distance means more similar vectors.
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating knowledge base
Loading embedding model and creating index


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2109.32it/s]


Knowledge base embeddings shape: (2000, 384)
Knowledge base successfully created


Zero-shot classifier for Q1, Q2, Q6


In [ ]:
import torch
print(torch.__version__)

2.5.1+cu121


In case need tensor flow

In [ ]:
!pip install tf-keras

Defaulting to user installation because normal site-packages is not writeable


In [10]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)

try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__)
except:
    print("TensorFlow not installed")

try:
    import keras
    print("Keras:", keras.__version__)
except:
    print("Keras not installed")

Transformers: 5.13.0
Torch: 2.5.1+cu121


I0000 00:00:1783658963.948831   32975 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783658965.105518   32975 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/studinstru/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warni

TensorFlow: 2.21.0
Keras: 3.12.0


Update transformer to the recent version

In [ ]:
pip install -U transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [11]:
zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
)

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 1343.61it/s]


In [12]:
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])
print("Prompt:", prompt_150)
print("Labels:", labels_150)
print("Answer:", ans_150)

Prompt: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.
Labels: ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical framework has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has

Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

In [13]:
result = zs(prompt_150, candidate_labels=labels_150)

print(result)

score = result["scores"][result["labels"].index(ans_150)]

print(f"Ans: Score for the correct answer {round(score, 3)}")

{'sequence': 'Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.', 'labels': ['The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."', 'The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."', 'The butterfly effect is the phenomenon that a small change in the initial conditio

Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [14]:
query = str(train.iloc[150]['prompt'])
query_embedding = model.encode([query], convert_to_numpy=True)

# search for top 10
distances, indices = index.search(query_embedding, k=10)

# Top-10 indices returned by FAISS in Q2
retrieved_indices = indices[0]
print(f"returned indices: {retrieved_indices}")
if 150 in retrieved_indices:
    rank = list(retrieved_indices).index(150) + 1
    print(f"Correct answer found at rank {rank} in the top 10 results.")
else:
    print("Correct answer not found in the top 10 results.")

returned indices: [ 663 1701 1269 1532  576  847 1693 1906  168  150]
Correct answer found at rank 10 in the top 10 results.


Concept: The Two-Stage Pipeline (Reranking & Cross-Encoders)
In Question 2, you saw that our FAISS database did not put the true document at rank #1. Why? Because FAISS uses Bi-encoder. 
A Bi-encoder embeds the question and the document separately and just compares the distance (Cosine Similarity). They are fast and allows you to search millions of documents but often miss semantic context. 

To fix this we use a 2 stage pipeline:
1. Retrieval: Use a bi-encoder with FAISS to quickly get the top k possible chunks/documents.
2. Reranking: Use a Cross-Encoder to deeply evaluate those top candidates and sort them based on highest semantic similarity.

A Cross-Encoder passes the Question and the Document into the Transformer network at the exact same time. The Attention mechanism can directly compare the words in the question to the words in the document, resulting in a highly accurate relevance score.

For this we part the prompt and the context and ask it to predict a score.

Cross Encoders (Hugging Face)

In [15]:
from sentence_transformers import CrossEncoder

# Load a pre-trained CrossEncoder model
model_test = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Predict scores for a pair of sentences
scores = model_test.predict([
    ("How many people live in Berlin?", "Berlin had a population of 3,520,031 registered inhabitants in an area of 891.82 square kilometers."),
    ("How many people live in Berlin?", "Berlin is well known for its museums."),
])

print(scores)
# => array([ 8.607138 , -4.3200774], dtype=float32)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1445.91it/s]


[ 8.607138 -4.320079]


Code to use a Cross-Encoder

In [16]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
# Retrieve corresponding documents
docs_10 = [kb[i] for i in retrieved_indices] #Get the top 10 chunks

# Create (question, document) pairs
pairs = [[prompt_150, doc] for doc in docs_10] #Create prompt-context pairs

# Compute Cross-Encoder scores
ce_scores = cross_encoder.predict(pairs) # Get the score of each pair

print("Cross-Encoder scores:", ce_scores)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3651.55it/s]


Cross-Encoder scores: [4.6602297 4.6602297 4.737521  4.737521  4.687044  4.752575  4.752575
 4.752575  4.7072463 4.7585096]


Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [17]:
# combine everything into a list of tuples for easier analysis
results = list(zip(retrieved_indices, docs_10, ce_scores))

# Sort by score (highest first)
results = sorted(results, key=lambda x: x[2], reverse=True)

# print ranking
for rank, (index, doc, score) in enumerate(results, start=1):
    print(f"Rank {rank}: KB Index {index}, Score {score:.4f}")
    #print(f"Document: {doc}\n")

correct_doc_index = 150

for rank, (idx, doc, score) in enumerate(results, start=1):
    if idx == correct_doc_index:
        print(f"Correct answer found at rank {rank} with score {score:.4f}")
        break

Rank 1: KB Index 150, Score 4.7585
Rank 2: KB Index 847, Score 4.7526
Rank 3: KB Index 1693, Score 4.7526
Rank 4: KB Index 1906, Score 4.7526
Rank 5: KB Index 1269, Score 4.7375
Rank 6: KB Index 1532, Score 4.7375
Rank 7: KB Index 168, Score 4.7072
Rank 8: KB Index 576, Score 4.6870
Rank 9: KB Index 663, Score 4.6602
Rank 10: KB Index 1701, Score 4.6602
Correct answer found at rank 1 with score 4.7585


Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

In [18]:
# "Flat" means store all vectors exactly as they are (no compression or approximation).
# "L2" means similarity is measured using the Euclidean (L2) distance
# Smaller distance means more similar vectors.
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

# prompt at row 42
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])

# embedding the prompt
query_embedding_42 = model.encode([prompt_42])

# Retrive top 5 documents
distance, indices = index.search(query_embedding_42, k=5)

retrieved_indices = indices[0]
docs_5 = [kb[i] for i in retrieved_indices] 

# Concatenate with a single space
concatenated_docs = " ".join(docs_5)

# Build the input string
text = f"Context: {concatenated_docs} Question: {prompt_42}"

# Tokenize without truncation
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

encoding = tokenizer(text, truncation=False)

# Count total tokens
num_tokens = len(encoding["input_ids"])

print("Total tokens:", num_tokens)


Total tokens: 216


Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).
*


In [25]:
row_150 = train.iloc[150]

prompt_150 = str(row_150['prompt'])

true_document = kb[150]

rag_input = f"Context: {true_document} Question: {prompt_150}"

labels = [
    str(row_150['A']),
    str(row_150['B']),
    str(row_150['C']),
    str(row_150['D']),
    str(row_150['E'])
]

# run the zero shot classification pipeline
result = zs(
    rag_input,
    candidate_labels=labels
)

correct_answer = str(row_150[row_150['answer']])

score = result["scores"][result["labels"].index(correct_answer)]

print(f"Score: {round(score, 3)}")

Score: 0.989


#### Concept: The Danger of Bad Retrieval (Adversarial RAG) 
The golden rule of Retrieval-Augmented Generation is “Garbage In, Garbage Out.” An LLM places immense trust in the external context you inject into its prompt. If your vector database performs poorly and retrieves an irrelevant or incorrect document, the model will often abandon its own internal reasoning and confidently generate the wrong answer based on that bad data. We do the reranking and constricting the number of chunks that we give to the model for the same reason.

Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).
*


In [29]:
row_150 = train.iloc[150]

prompt_150 = str(row_150["prompt"])
correct_answer = str(row_150[row_150["answer"]])

# Assuming 999 is a wrong document index for demonstration
wrong_document = kb[999]  

adversarial_rag = (
    f"Context: {wrong_document} Question: {prompt_150}"
)

labels = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]

result_adversarial = zs(
    adversarial_rag,
    candidate_labels=labels
)

result_adversarial_score = result_adversarial["scores"][result_adversarial["labels"].index(correct_answer)]

print(f"Probability score for the correct answer with adversarial context: {round(result_adversarial_score, 3)}")


Probability score for the correct answer with adversarial context: 0.529


In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.

Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

In [31]:
hits = 0

for i in range(100):
    # current row
    row = train.iloc[i]
    
    # prompt
    prompt = str(row["prompt"])
    
    # correct answer
    correct_answer = str(row[row["answer"]])

    # embed the prompt
    query_embedding = model.encode([prompt])

    # retrieve top 5 documents
    distance, indices = index.search(query_embedding, k=5)

    retrieved_indices = indices[0]

    # retrieve the corresponding documents
    docs_5 = [kb[i] for i in retrieved_indices]

    if correct_answer in docs_5:
        hits += 1

hit_rate = (hits / 100) * 100

print(f"Number of hits in the first 100 prompts: {hits}")
print(f"Hit rate for the first 100 prompts: {hit_rate:.2f}%")



Number of hits in the first 100 prompts: 73
Hit rate for the first 100 prompts: 73.00%


Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.

For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [32]:

def map_at_3(actual, predicted):
    """
    actual: correct option letter, e.g. 'C'
    predicted: list of top-3 predicted option letters
    """
    for i, p in enumerate(predicted):
        if p == actual:
            return 1.0 / (i + 1)
    return 0.0
    
scores = []
for i in range(20):
    row = train.iloc[i]
    prompt = str(row["prompt"])
    
    # Retrieve (FAISS)
    query_embedding = model.encode([prompt])
    distance, indices = index.search(query_embedding, k=5)
    retrieved_indices = indices[0]
    docs_5 = [kb[i] for i in retrieved_indices]

    # Rerank (Cross-Encoder)
    pairs = [[prompt, doc] for doc in docs_5]
    ce_scores = cross_encoder.predict(pairs)
    best_doc = docs_5[np.argmax(ce_scores)]

    # RAG (Zero-Shot Classification)
    rag_prompt = f"Context: {best_doc} Question: {prompt}"
    labels = [
        str(row["A"]),
        str(row["B"]),
        str(row["C"]),
        str(row["D"]),
        str(row["E"])
    ]
    
    # Zero-Shot Classification
    result  = zs(rag_prompt, candidate_labels=labels)

    correct_option = row["answer"] # correct option from the row

    predicted_options = []
    for option_text in result["labels"][:3]: # Get top 3 predicted options
        for option in ["A", "B", "C", "D", "E"]:
            if str(row[option]) == option_text:
                predicted_options.append(option)
                break
            

    # MAP@3
    score = map_at_3(correct_option, predicted_options)
    scores.append(score)

# Final MAP@3 score
final_map_at_3 = np.mean(scores)
print(f"Final MAP@3 score for the first 20 prompts: {round(final_map_at_3, 3)}")


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Final MAP@3 score for the first 20 prompts: 0.975
